# Microsoft Agent Framework — Azure OpenAI（Responses API）

このコードサンプルでは、**Microsoft Agent Framework (MAF)** を使用して、**Responses API** を使う **Azure OpenAI** によってバックアップされた簡単なエージェントを作成します。

> **移行に関する注意:** このサンプルは以前、Semantic Kernel と GitHub Models を使用していました。Microsoft Agent Framework に移行し、GitHub Models（廃止予定、2026年7月に終了）は Responses API をサポートする Azure OpenAI に置き換えられました。MAF の `OpenAIChatClient` は Azure OpenAI の安定した `/openai/v1/` エンドポイントをターゲットにし、デフォルトで Responses API を使用します。

このサンプルの目的は、後にさまざまなエージェントパターンを実装する際に適用される手順を示すことです。


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## 必要なPythonパッケージのインポート


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ツールの定義

Microsoft Agent Framework において、<strong>ツール</strong>とはエージェントが呼び出せる `@tool` でデコレートされた単純な Python 関数です。以下では、ランダムな旅行先を返し、前回と同じ場所を繰り返さないツールを定義します。


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## エージェントの作成

ここでは、`TravelAgent` という名前のエージェントを作成します。

この例では非常に基本的な指示を使用しています。エージェントの挙動がどのように変わるか観察するために、これらの指示を自由に変更してください。


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## エージェントの実行

これでエージェントを実行できます。エージェントがターンを超えて会話を記憶できるように、`AgentSession` を作成し、2つの `user_inputs` を送信します。最初は旅行のリクエスト、2つ目は提案が気に入らなかったと言い別の提案を求める内容です — エージェントはセッション履歴と `get_random_destination` ツールを使って応答します。

これらのメッセージを変更して、エージェントがどのように反応を変えるか観察できます。応答はトークン単位で<strong>ストリーミング</strong>されます。


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
